# 1. Setup and File Paths

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

In [2]:
current_dir = Path.cwd().resolve()
project_name = 'ride-hailing-demand-fleet-allocation'

if current_dir.name == project_name:
    project_root = current_dir
elif current_dir.name == 'notebooks' and current_dir.parent.name == project_name:
    project_root = current_dir.parent
else:
    project_root = current_dir / project_name

if not project_root.exists():
    raise FileNotFoundError(f'Project folder not found: {project_root}')

raw_dir = project_root / 'data' / 'raw'

In [5]:
trip_files = sorted(raw_dir.glob('fhvhv_tripdata_2024-*.parquet'))
zone_file = raw_dir / 'taxi_zone_lookup.csv'
parquet_pattern = (raw_dir / 'fhvhv_tripdata_2024-*.parquet').as_posix()

con = duckdb.connect()
print(f'Project root       : {project_root}')
print(f'Trip files found   : {len(trip_files)}')
print(f'Zone lookup exists : {zone_file.exists()}')

Project root       : D:\My Journey\Data Project\CLAUDE\ride-hailing-demand-fleet-allocation
Trip files found   : 12
Zone lookup exists : True


# 2. File Coverage and Schema Consistency

In [6]:
df_file_coverage = con.execute(f"""
    SELECT filename, COUNT(*) AS total_rows
    FROM read_parquet('{parquet_pattern}', filename=true)
    GROUP BY filename
    ORDER BY filename
""").df()

df_file_coverage

,filename,total_rows
0,D:\My Journey\Data Project\CLAUDE\ride-hailing...,19663930
1,D:\My Journey\Data Project\CLAUDE\ride-hailing...,19359148
2,D:\My Journey\Data Project\CLAUDE\ride-hailing...,21280788
3,D:\My Journey\Data Project\CLAUDE\ride-hailing...,19733038
4,D:\My Journey\Data Project\CLAUDE\ride-hailing...,20704538
5,D:\My Journey\Data Project\CLAUDE\ride-hailing...,20123226
6,D:\My Journey\Data Project\CLAUDE\ride-hailing...,19182934
7,D:\My Journey\Data Project\CLAUDE\ride-hailing...,19128392
8,D:\My Journey\Data Project\CLAUDE\ride-hailing...,19209788
9,D:\My Journey\Data Project\CLAUDE\ride-hailing...,20028282


In [7]:
file_count = len(df_file_coverage)
total_rows = df_file_coverage['total_rows'].sum()

print(f'Files audited : {file_count}')
print(f'Total rows    : {total_rows:,}')

Files audited : 12
Total rows    : 239,470,448


In [10]:
df_schema_check = con.execute(f"""
    SELECT
        name AS column_name,
        type AS column_type,
        COUNT(DISTINCT file_name) AS files_found
    FROM parquet_schema('{parquet_pattern}')
    WHERE type IS NOT NULL
    GROUP BY name, type
    ORDER BY name
""").df()

schema_rows = len(df_schema_check)
consistent_columns = (df_schema_check['files_found'] == 12).sum()

display(df_schema_check)
print(f'Columns found      : {schema_rows}')
print(f'Consistent columns : {consistent_columns}')

,column_name,column_type,files_found
0,DOLocationID,INT32,12
1,PULocationID,INT32,12
2,access_a_ride_flag,BYTE_ARRAY,12
3,airport_fee,DOUBLE,12
4,base_passenger_fare,DOUBLE,12
5,bcf,DOUBLE,12
6,congestion_surcharge,DOUBLE,12
7,dispatching_base_num,BYTE_ARRAY,12
8,driver_pay,DOUBLE,12
9,dropoff_datetime,INT64,12


Columns found      : 24
Consistent columns : 24


# 3. Timestamp and Zone Validation

In [12]:
df_timestamp_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(pickup_datetime) AS min_pickup,
        MAX(pickup_datetime) AS max_pickup,
        COUNT(*) FILTER (
            WHERE pickup_datetime IS NULL
        ) AS null_pickup,
        COUNT(*) FILTER (
            WHERE dropoff_datetime IS NULL
        ) AS null_dropoff,
        COUNT(*) FILTER (
            WHERE dropoff_datetime <= pickup_datetime
        ) AS invalid_time_order,
        COUNT(*) FILTER (
            WHERE pickup_datetime < TIMESTAMP '2024-01-01'
            OR pickup_datetime >= TIMESTAMP '2025-01-01'
        ) AS pickup_outside_2024
    FROM read_parquet('{parquet_pattern}')
""").df()

df_timestamp_check

,total_rows,min_pickup,max_pickup,null_pickup,null_dropoff,invalid_time_order,pickup_outside_2024
0,239470448,2024-01-01,2024-12-31 23:59:59,0,0,9680,0


In [25]:
df_zone_lookup_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_zones,
        COUNT(DISTINCT LocationID) AS unique_zone_ids
    FROM read_csv_auto('{zone_file.as_posix()}')
""").df()

total_timestamp_rows = df_timestamp_check.loc[0, 'total_rows']
invalid_time_rows = df_timestamp_check.loc[0, 'invalid_time_order']
invalid_time_percent = invalid_time_rows / total_timestamp_rows * 100

display(df_zone_lookup_check)
print(f'Invalid time rows    : {invalid_time_rows:,}')
print(f'Invalid time percent : {invalid_time_percent:.6f}%')

,total_zones,unique_zone_ids
0,265,265


Invalid time rows    : 9,680
Invalid time percent : 0.004042%


In [16]:
df_zone_check = con.execute(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE trip.PULocationID IS NULL
        ) AS null_pickup_zone,
        COUNT(*) FILTER (
            WHERE trip.DOLocationID IS NULL
        ) AS null_dropoff_zone,
        COUNT(*) FILTER (
            WHERE trip.PULocationID IS NOT NULL
            AND pickup_zone.LocationID IS NULL
        ) AS invalid_pickup_zone,
        COUNT(*) FILTER (
            WHERE trip.DOLocationID IS NOT NULL
            AND dropoff_zone.LocationID IS NULL
        ) AS invalid_dropoff_zone
    FROM read_parquet('{parquet_pattern}') AS trip
    LEFT JOIN read_csv_auto('{zone_file.as_posix()}') AS pickup_zone
        ON trip.PULocationID = pickup_zone.LocationID
    LEFT JOIN read_csv_auto('{zone_file.as_posix()}') AS dropoff_zone
        ON trip.DOLocationID = dropoff_zone.LocationID
""").df()

df_zone_check

,null_pickup_zone,null_dropoff_zone,invalid_pickup_zone,invalid_dropoff_zone
0,0,0,0,0


# 4. Distance, Duration, and Missing Values

In [17]:
df_distance_check = con.execute(f"""
    SELECT
        MIN(trip_miles) AS min_miles,
        APPROX_QUANTILE(trip_miles, 0.50) AS median_miles,
        APPROX_QUANTILE(trip_miles, 0.99) AS p99_miles,
        MAX(trip_miles) AS max_miles,
        COUNT(*) FILTER (
            WHERE trip_miles <= 0
        ) AS invalid_miles
    FROM read_parquet('{parquet_pattern}')
""").df()

invalid_miles = df_distance_check.loc[0, 'invalid_miles']
invalid_miles_percent = invalid_miles / total_rows * 100

display(df_distance_check)
print(f'Invalid distance rows    : {invalid_miles:,}')
print(f'Invalid distance percent : {invalid_miles_percent:.6f}%')

,min_miles,median_miles,p99_miles,max_miles,invalid_miles
0,0.0,3.014522,27.386048,555.25,34059


Invalid distance rows    : 34,059
Invalid distance percent : 0.014223%


In [21]:
df_extreme_trips = con.execute(f"""
    SELECT
        pickup_datetime,
        dropoff_datetime,
        PULocationID,
        DOLocationID,
        trip_miles,
        trip_time,
        ROUND(
            trip_miles / (trip_time / 3600.0),
            2
        ) AS average_speed_mph
    FROM read_parquet('{parquet_pattern}')
    WHERE trip_miles > 0 AND trip_time > 0
    ORDER BY trip_miles DESC
    LIMIT 10
""").df()

df_extreme_trips

,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,trip_miles,trip_time,average_speed_mph
0,2024-12-17 20:47:24,2024-12-18 04:57:26,124,265,555.250,29402,67.99
1,2024-12-28 21:17:49,2024-12-29 05:31:47,22,229,490.310,29638,59.56
2,2024-08-24 13:45:40,2024-08-24 21:57:01,232,229,488.662,29481,59.67
3,2024-09-09 19:52:27,2024-09-10 02:57:25,239,265,469.408,25498,66.27
4,2024-10-04 00:50:55,2024-10-04 07:22:11,224,180,455.520,23476,69.85
5,2024-01-16 01:36:36,2024-01-16 11:06:17,186,265,417.620,34181,43.98
6,2024-12-09 03:43:04,2024-12-09 11:35:28,92,138,394.890,28344,50.16
7,2024-12-11 07:54:32,2024-12-11 16:04:34,41,265,373.730,29402,45.76
8,2024-01-09 13:04:08,2024-01-09 21:32:12,48,265,367.000,30484,43.34
9,2024-06-30 02:48:09,2024-06-30 08:33:40,132,265,363.550,20731,63.13


In [19]:
df_duration_check = con.execute(f"""
    SELECT
        MIN(trip_time) AS min_seconds,
        APPROX_QUANTILE(trip_time, 0.50) AS median_seconds,
        APPROX_QUANTILE(trip_time, 0.99) AS p99_seconds,
        MAX(trip_time) AS max_seconds,
        COUNT(*) FILTER (
            WHERE trip_time <= 0
        ) AS invalid_duration
    FROM read_parquet('{parquet_pattern}')
""").df()

invalid_duration = df_duration_check.loc[0, 'invalid_duration']
invalid_duration_percent = invalid_duration / total_rows * 100

display(df_duration_check)
print(f'Invalid duration rows    : {invalid_duration:,}')
print(f'Invalid duration percent : {invalid_duration_percent:.6f}%')

,min_seconds,median_seconds,p99_seconds,max_seconds,invalid_duration
0,0,978,4308,55138,30


Invalid duration rows    : 30
Invalid duration percent : 0.000013%


In [20]:
df_missing_check = con.execute(f"""
    SELECT
        COUNT(*) FILTER (
            WHERE hvfhs_license_num IS NULL
        ) AS null_license,
        COUNT(*) FILTER (
            WHERE dispatching_base_num IS NULL
        ) AS null_dispatching_base,
        COUNT(*) FILTER (
            WHERE trip_miles IS NULL
        ) AS null_trip_miles,
        COUNT(*) FILTER (
            WHERE trip_time IS NULL
        ) AS null_trip_time
    FROM read_parquet('{parquet_pattern}')
""").df()

df_missing_check

,null_license,null_dispatching_base,null_trip_miles,null_trip_time
0,0,0,0,0


# 5. Duplicate Detection

In [22]:
df_duplicate_check = con.execute(f"""
    WITH duplicate_groups AS (
        SELECT
            dispatching_base_num,
            pickup_datetime,
            dropoff_datetime,
            PULocationID,
            DOLocationID,
            COUNT(*) AS rows_per_group
        FROM read_parquet('{parquet_pattern}')
        GROUP BY
            dispatching_base_num,
            pickup_datetime,
            dropoff_datetime,
            PULocationID,
            DOLocationID
        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS duplicate_groups,
        SUM(rows_per_group) AS rows_in_duplicate_groups,
        SUM(rows_per_group - 1) AS extra_duplicate_rows
    FROM duplicate_groups
""").df()

df_duplicate_check

,duplicate_groups,rows_in_duplicate_groups,extra_duplicate_rows
0,439,879.0,440.0


In [23]:
df_exact_duplicate_check = con.execute(f"""
    WITH exact_duplicate_groups AS (
        SELECT
            *,
            COUNT(*) AS rows_per_group
        FROM read_parquet('{parquet_pattern}')
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )

    SELECT
        COUNT(*) AS exact_duplicate_groups,
        SUM(rows_per_group) AS rows_in_exact_groups,
        SUM(rows_per_group - 1) AS exact_duplicate_rows
    FROM exact_duplicate_groups
""").df()

df_exact_duplicate_check

,exact_duplicate_groups,rows_in_exact_groups,exact_duplicate_rows
0,0,NaN,NaN


# 6. Audit Summary and Cleaning Rules

In [27]:
df_cleaning_summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE dropoff_datetime <= pickup_datetime
        ) AS invalid_timestamp,
        COUNT(*) FILTER (
            WHERE trip_miles <= 0
        ) AS invalid_distance,
        COUNT(*) FILTER (
            WHERE trip_time <= 0
        ) AS invalid_duration,
        COUNT(*) FILTER (
            WHERE dropoff_datetime <= pickup_datetime
            OR trip_miles <= 0
            OR trip_time <= 0
        ) AS rows_to_remove,
        COUNT(*) FILTER (
            WHERE dropoff_datetime > pickup_datetime
            AND trip_miles > 0
            AND trip_time > 0
        ) AS rows_to_keep
    FROM read_parquet('{parquet_pattern}')
""").df()

total_rows = df_cleaning_summary.loc[0, 'total_rows']
rows_to_remove = df_cleaning_summary.loc[0, 'rows_to_remove']
rows_to_keep = df_cleaning_summary.loc[0, 'rows_to_keep']

remove_percent = rows_to_remove / total_rows * 100
keep_percent = rows_to_keep / total_rows * 100

display(df_cleaning_summary)
print(f'Rows to remove : {rows_to_remove:,} ({remove_percent:.6f}%)')
print(f'Rows to keep   : {rows_to_keep:,} ({keep_percent:.6f}%)')

,total_rows,invalid_timestamp,invalid_distance,invalid_duration,rows_to_remove,rows_to_keep
0,239470448,9680,34059,30,43711,239426737


Rows to remove : 43,711 (0.018253%)
Rows to keep   : 239,426,737 (99.981747%)


In [28]:
schema_issues = schema_rows - consistent_columns
zone_issues = int(df_zone_check.iloc[0].sum())
missing_issues = int(df_missing_check.iloc[0].sum())

potential_duplicate_rows = int(
    df_duplicate_check.loc[0, 'extra_duplicate_rows']
)

exact_duplicate_rows = df_exact_duplicate_check.loc[
    0,
    'exact_duplicate_rows'
]

if pd.isna(exact_duplicate_rows):
    exact_duplicate_rows = 0
else:
    exact_duplicate_rows = int(exact_duplicate_rows)

In [29]:
audit_summary = pd.DataFrame({
    'check': [
        'schema_inconsistency',
        'invalid_timestamp',
        'invalid_zone',
        'invalid_distance',
        'invalid_duration',
        'missing_critical_values',
        'potential_duplicate',
        'exact_duplicate'
    ],
    'affected_rows': [
        schema_issues,
        invalid_time_rows,
        zone_issues,
        invalid_miles,
        invalid_duration,
        missing_issues,
        potential_duplicate_rows,
        exact_duplicate_rows
    ],
    'decision': [
        'no_action',
        'remove',
        'no_action',
        'remove',
        'remove',
        'no_action',
        'retain_not_exact',
        'no_action'
    ]
})

audit_summary

,check,affected_rows,decision
0,schema_inconsistency,0,no_action
1,invalid_timestamp,9680,remove
2,invalid_zone,0,no_action
3,invalid_distance,34059,remove
4,invalid_duration,30,remove
5,missing_critical_values,0,no_action
6,potential_duplicate,440,retain_not_exact
7,exact_duplicate,0,no_action


In [32]:
audit_path = (
    project_root
    / 'data'
    / 'manifests'
    / 'data_quality_audit_2024.csv'
)

audit_path.parent.mkdir(parents=True, exist_ok=True)
audit_summary.to_csv(audit_path, index=False)

print(f'Audit summary saved to: {audit_path}')
print()
print('Cleaning rules for Notebook 03:')
print('- Keep dropoff_datetime > pickup_datetime')
print('- Keep trip_miles > 0')
print('- Keep trip_time > 0')
print('- Retain positive extreme values')
print('- Do not remove potential duplicates')

Audit summary saved to: D:\My Journey\Data Project\CLAUDE\ride-hailing-demand-fleet-allocation\data\manifests\data_quality_audit_2024.csv

Cleaning rules for Notebook 03:
- Keep dropoff_datetime > pickup_datetime
- Keep trip_miles > 0
- Keep trip_time > 0
- Retain positive extreme values
- Do not remove potential duplicates


In [33]:
con.close()